# 2.b Translation eval — 预测 vs 真值

比较用户模型输出与 `work/GT_fasta/` 真值序列（**绘图在 `3.plot.ipynb`**）：

| 任务 | 预测 | 真值 |
|------|------|------|
| **AA→3Di** | `work/aa2di_fasta/*` | `GT_fasta/DB_di.fasta` |
| **3Di→AA** | `work/di2aa_fasta/*` | `GT_fasta/DB_aa.fasta` |

指标：micro / macro accuracy、exact match；输出 `work/metrics/translation/`。

可与 `1.init` 独立运行。共享配置：[`config.py`](config.py)。


In [5]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from config import (
    AA2DI_FASTA_DIR,
    AA_FASTA,
    DI2AA_FASTA_DIR,
    GT_DI_FASTA,
    PROJECT_ROOT,
    STANDARD_AA,
    TRANSLATION_METHODS,
    TRANSLATION_METRICS_DIR,
    cleanup_tmp,
    ensure_work_dirs,
    require_project_root,
    translation_per_seq_path,
    translation_summary_path,
)

ROOT = require_project_root("2.b.translation_eval.ipynb")
assert ROOT == PROJECT_ROOT

SKIP_EXISTING = True
ONLY_METHODS: list[str] | None = None  # 例如 ["ESM3", "ProstT5"]
SKIP_LENGTH_MISMATCH = True  # True = 跳过长度不一致的序列

ensure_work_dirs()
print("ROOT:", ROOT)
print("GT aa:", AA_FASTA)
print("GT di:", GT_DI_FASTA)
print("out:", TRANSLATION_METRICS_DIR)


ROOT: /hpcfs/fhome/caihuize/scope40_easy
GT aa: /hpcfs/fhome/caihuize/scope40_easy/work/GT_fasta/DB_aa.fasta
GT di: /hpcfs/fhome/caihuize/scope40_easy/work/GT_fasta/DB_di.fasta
out: /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation


In [6]:
def load_fasta(path: Path) -> dict[str, str]:
    """Minimal FASTA reader — no Biopython required."""
    out: dict[str, str] = {}
    name: str | None = None
    chunks: list[str] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    out[name] = "".join(chunks)
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line.strip())
        if name is not None:
            out[name] = "".join(chunks)
    return out


@dataclass
class SeqMetrics:
    qid: str
    length: int
    acc: float
    exact_match: int
    n_mismatch: int


@dataclass
class TaskSummary:
    task: str
    method_key: str
    label: str
    n_seqs: int
    n_residues: int
    micro_acc: float
    macro_acc: float
    exact_match_frac: float
    length_mismatch_n: int
    missing_in_pred_n: int
    invalid_char_frac: float


def _compare_sequences(
    pred: str,
    gt: str,
    *,
    uppercase: bool,
    valid_chars: set[str] | None,
) -> tuple[float, int, int, float]:
    if uppercase:
        pred, gt = pred.upper(), gt.upper()
    if len(pred) != len(gt):
        raise ValueError("length mismatch")
    if len(pred) == 0:
        return 1.0, 1, 0, 0.0
    correct = sum(p == g for p, g in zip(pred, gt, strict=True))
    acc = correct / len(pred)
    exact = int(pred == gt)
    invalid = 0
    if valid_chars is not None:
        invalid = sum(1 for p in pred if p not in valid_chars)
    invalid_frac = invalid / len(pred)
    return acc, exact, len(pred) - correct, invalid_frac


def evaluate_task(
    task: str,
    method_key: str,
    label: str,
    pred_path: Path,
    gt_path: Path,
    *,
    uppercase: bool,
    valid_chars: set[str] | None,
    skip_existing: bool = True,
) -> TaskSummary | None:
    per_seq_out = translation_per_seq_path(task, method_key)
    if skip_existing and per_seq_out.is_file() and per_seq_out.stat().st_size > 0:
        print(f"⏭️  已存在: {per_seq_out}")
        row = pd.read_csv(per_seq_out, sep="\t")
        if row.empty:
            return None
        return TaskSummary(
            task=task,
            method_key=method_key,
            label=label,
            n_seqs=len(row),
            n_residues=int(row["length"].sum()),
            micro_acc=float(row["n_match"].sum() / row["length"].sum()) if row["length"].sum() else 0.0,
            macro_acc=float(row["acc"].mean()),
            exact_match_frac=float(row["exact_match"].mean()),
            length_mismatch_n=0,
            missing_in_pred_n=0,
            invalid_char_frac=float(row.get("invalid_char_frac", pd.Series([0.0])).mean()),
        )

    if not pred_path.is_file():
        print(f"⏭️  跳过（无预测文件）: {pred_path}")
        return None
    if not gt_path.is_file():
        raise FileNotFoundError(f"缺少真值: {gt_path}")

    pred = load_fasta(pred_path)
    gt = load_fasta(gt_path)
    gt_ids = set(gt)
    pred_ids = set(pred)
    missing = gt_ids - pred_ids
    extra = pred_ids - gt_ids
    if extra:
        print(f"⚠️  {label} {task}: 预测多 {len(extra)} 个 ID（将忽略）")
    if missing:
        print(f"⚠️  {label} {task}: 真值中 {len(missing)} 个 ID 无预测")

    rows: list[dict] = []
    length_mismatch = 0
    total_correct = 0
    total_len = 0
    invalid_fracs: list[float] = []

    for qid in sorted(gt_ids & pred_ids):
        p, g = pred[qid], gt[qid]
        if len(p) != len(g):
            length_mismatch += 1
            if SKIP_LENGTH_MISMATCH:
                continue
            n = min(len(p), len(g))
            p, g = p[:n], g[:n]
        n = len(g)
        acc, exact, n_mis, inv_frac = _compare_sequences(
            p, g, uppercase=uppercase, valid_chars=valid_chars
        )
        n_match = n - n_mis
        rows.append(
            {
                "qid": qid,
                "length": n,
                "acc": acc,
                "exact_match": exact,
                "n_mismatch": n_mis,
                "n_match": n_match,
                "invalid_char_frac": inv_frac,
            }
        )
        total_correct += n_match
        total_len += n
        invalid_fracs.append(inv_frac)

    if not rows:
        print(f"❌ {label} {task}: 无有效可比序列")
        return None

    df = pd.DataFrame(rows)
    per_seq_out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(per_seq_out, sep="\t", index=False)
    print(f"✅ {per_seq_out}  n={len(df)}")

    return TaskSummary(
        task=task,
        method_key=method_key,
        label=label,
        n_seqs=len(df),
        n_residues=total_len,
        micro_acc=total_correct / total_len if total_len else 0.0,
        macro_acc=float(df["acc"].mean()),
        exact_match_frac=float(df["exact_match"].mean()),
        length_mismatch_n=length_mismatch,
        missing_in_pred_n=len(missing),
        invalid_char_frac=float(sum(invalid_fracs) / len(invalid_fracs)) if invalid_fracs else 0.0,
    )


def run_all(skip_existing: bool = True) -> pd.DataFrame:
    summaries: list[TaskSummary] = []
    methods = TRANSLATION_METHODS
    if ONLY_METHODS is not None:
        methods = [m for m in methods if m[1] in ONLY_METHODS]

    gt_di_chars: set[str] | None = None
    if GT_DI_FASTA.is_file():
        gt_di_chars = set("".join(load_fasta(GT_DI_FASTA).values()).upper())

    for label, key, aa2di_name, di2aa_name in methods:
        print(f"\n══ {label} ══")
        s_aa2di = evaluate_task(
            "aa2di",
            key,
            label,
            AA2DI_FASTA_DIR / aa2di_name,
            GT_DI_FASTA,
            uppercase=True,
            valid_chars=gt_di_chars,
            skip_existing=skip_existing,
        )
        if s_aa2di is not None:
            summaries.append(s_aa2di)
        s_di2aa = evaluate_task(
            "di2aa",
            key,
            label,
            DI2AA_FASTA_DIR / di2aa_name,
            AA_FASTA,
            uppercase=False,
            valid_chars=STANDARD_AA,
            skip_existing=skip_existing,
        )
        if s_di2aa is not None:
            summaries.append(s_di2aa)

    if not summaries:
        return pd.DataFrame()

    df = pd.DataFrame([s.__dict__ for s in summaries])
    for task in ("aa2di", "di2aa"):
        sub = df[df["task"] == task]
        if not sub.empty:
            path = translation_summary_path(task)
            sub.to_csv(path, index=False)
            print(f"\n✅ {path}")
    combined = TRANSLATION_METRICS_DIR / "translation_summary.csv"
    df.to_csv(combined, index=False)
    print(f"✅ {combined}")
    return df


summary_df = run_all(skip_existing=SKIP_EXISTING)
display(summary_df)
print("\nTranslation eval 完成。下一步打开 3.plot.ipynb（作图）")



══ ESM3-3Di ══
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/aa2di_ESM3_per_seq.tsv  n=13907
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/di2aa_ESM3_per_seq.tsv  n=13920

══ ESM3-LoRA ══
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/aa2di_ESM3_LoRA_per_seq.tsv  n=13920
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/di2aa_ESM3_LoRA_per_seq.tsv  n=13920

══ ProstT5 (translate) ══
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/aa2di_ProstT5_per_seq.tsv  n=13917
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/di2aa_ProstT5_per_seq.tsv  n=13917

══ SaProt ══
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/aa2di_SaProt_per_seq.tsv  n=13920
✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/di2aa_SaProt_per_seq.tsv  n=13920

✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/aa2di_summary.csv

✅ /hpcfs/fhome/caihuize/scope40_easy/work/metrics/translation/di2aa_summ

,task,method_key,label,n_seqs,n_residues,micro_acc,macro_acc,exact_match_frac,length_mismatch_n,missing_in_pred_n,invalid_char_frac
0,aa2di,ESM3,ESM3-3Di,13907,2647952,0.603997,0.613107,0.000431,13,0,0.000000
1,di2aa,ESM3,ESM3-3Di,13920,2652022,0.234293,0.237777,0.000000,0,0,0.000000
2,aa2di,ESM3_LoRA,ESM3-LoRA,13920,2652022,0.655801,0.661693,0.000503,0,0,0.000000
3,di2aa,ESM3_LoRA,ESM3-LoRA,13920,2652022,0.393107,0.375720,0.000000,0,0,0.000000
4,aa2di,ProstT5,ProstT5 (translate),13917,2651760,0.669783,0.676822,0.000431,3,0,0.000000
5,di2aa,ProstT5,ProstT5 (translate),13917,2651470,0.371902,0.354864,0.000000,3,0,0.000002
6,aa2di,SaProt,SaProt,13920,2652022,0.408566,0.434475,0.000216,0,0,0.000000
7,di2aa,SaProt,SaProt,13920,2652022,0.412799,0.397912,0.000000,0,0,0.000000



Translation eval 完成。下一步打开 3.plot.ipynb（作图）


## 清理临时目录

删除 `tmp/` 与 `work/tmp/`。


In [7]:
cleanup_tmp(also_work_tmp=True)


🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/tmp
🧹 已清理: /hpcfs/fhome/caihuize/scope40_easy/work/tmp
